# Local Healthcare RAG

Retrieval-augmented Q&A over a 4,114-page medical reference manual, running entirely on local infrastructure: **Ollama** for embeddings (`nomic-embed-text`) and generation (any local chat model), **Chroma** for the vector store. No cloud, no API keys.

Pipeline: PDF -> chunk -> embed (nomic) -> Chroma -> retrieve top-k -> grounded, cited answer. Logic in `src/`.

In [1]:
import sys, pathlib, os
_root = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent) if (p/'src').exists())
sys.path.insert(0, str(_root))
import ollama
from src import ingest, rag, eval as rageval
from src.config import COLLECTION, EMBED_MODEL

## Build (or reuse) the index

Indexes a 500-page subset for a fast demo. Full manual: `uv run python -m src.ingest`.

In [2]:
import chromadb
from src.config import CHROMA_DIR
try:
    n = chromadb.PersistentClient(path=CHROMA_DIR).get_collection(COLLECTION).count()
except Exception:
    n = 0
if n == 0:
    n = ingest.build_index(max_pages=500)
print(f'index ready: {n} chunks  (embeddings: {EMBED_MODEL})')

index ready: 2248 chunks  (embeddings: nomic-embed-text:v1.5)


## Retrieve

Top passages for a clinical query, with page citations and cosine distance.

In [3]:
for h in rag.retrieve('What causes diabetes in pancreatic disease?', k=3):
    print(f"p.{h['page']}  dist={h['distance']}")
    print('  ', h['text'][:140].strip(), '...\n')

p.277  dist=0.2047
   f chronic pancreatitis, and possibly long-standing diabetes mellitus (primarily in women). Heredity plays some role. Alcohol and caffeine co ...

p.209  dist=0.2363
   Chapter 15. Pancreatitis Introduction Pancreatitis is classified as either acute or chronic. Acute pancreatitis is inflammation that resolve ...

p.209  dist=0.2489
   mes, adjacent tissues) caused by the release of activated pancreatic enzymes. The most common triggers are biliary tract disease and chronic ...



## Grounded generation

The demo uses whatever local chat model is available. For real clinical use, configure a general or medical instruct model via `RAG_LLM_MODEL` (e.g. `ollama pull llama3.2`). Here it falls back to any non-embedding model present.

In [4]:
chat_models = [m.model for m in ollama.Client().list().models if 'embed' not in m.model]
demo_model = os.environ.get('RAG_LLM_MODEL') if rag.model_available(os.environ.get('RAG_LLM_MODEL','')) else (chat_models[0] if chat_models else None)
print('using chat model:', demo_model)
q = 'What can cause diabetes in patients with pancreatic disease?'
result = rag.answer(q, k=3, model=demo_model)
print('\nsources (pages):', result['sources'])
print('\nanswer:\n', result.get('answer') or result.get('note'))

using chat model: selah-q4:latest



sources (pages): [209, 215, 277]

answer:
 Pancreatic cancer causes diabetes in 25 to 50% of patients [p. 277]. Additionally, diabetes can be caused by chronic pancreatitis [p. 215].


## Self-evaluation

The model rates its own answer for groundedness and relevance (a cheap signal, not a human-graded test set).

In [5]:
if result.get('answer'):
    print(rageval.grade(q, result, model=demo_model))
else:
    print('skipped (no chat model available)')

{'groundedness': 5, 'relevance': 5, 'reason': "The answer is fully supported by the provided text. It correctly identifies pancreatic cancer as a cause (with the specific statistic from p. 277) and chronic pancreatitis as a cause (as mentioned in p. 215 and p. 209). It directly answers the user's question."}


## Scope & safety

This is a **reference** tool for clinicians grounded in a manual, not a source of individualized medical advice. The system prompt forces citation and refusal when the context lacks an answer. Everything runs locally; no patient data leaves the machine.